# Week 4 Lab: Associative Learning Models

This week formalizes how organisms learn predictive relationships between events.
The lab implements two associative-learning models as **recursive update rules**,
where the state at trial $t$ feeds into trial $t+1$:

- **Part 1 - Rescorla-Wagner.** Learning is driven by prediction error:
  $$\Delta V_i = \alpha_i\,\beta\,(\lambda - V_{total}), \qquad V_{total} = \sum_i V_i$$
  Use it to reproduce acquisition, blocking, overshadowing, overexpectation, and
  conditioned inhibition.
- **Part 2 - Mackintosh (1975).** Extends prediction-error learning with a dynamic
  *associability* (attention) term that itself updates across trials.

Code cells are left blank for you to fill in. Both models are pure simulations -
no data files are needed for Part 1.

## Setup

Import `numpy`, `pandas`, `matplotlib`, and `seaborn`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')

## Part 1: The Rescorla-Wagner Model

### Task 1: Implement the update rule and simulate acquisition

Write a function that takes a list of trials - each trial is the set of stimuli present plus the US asymptote `lambda` (1.0 when the US occurs, 0.0 when it does not) - and applies the Rescorla-Wagner rule $\Delta V_i = \alpha_i\,\beta\,(\lambda - V_{total})$, where $V_{total}$ sums associative strengths over the stimuli present on that trial. Use it to simulate simple acquisition of a single CS and plot $V$ across trials.

In [ ]:
def rescorla_wagner(trials, alphas, beta=0.3):
    """Simulate the Rescorla-Wagner model.

    trials : list of (present_stimuli, lam) - stimuli present and US asymptote on each trial
    alphas : dict {cs_name: salience alpha}
    beta   : US learning-rate parameter
    Returns a DataFrame with associative strength V for each CS across trials.
    """
    stim = list(alphas.keys())
    V = {s: 0.0 for s in stim}
    rows = []
    for t, (present, lam) in enumerate(trials, start=1):
        V_total = sum(V[s] for s in present)
        for s in present:
            V[s] += alphas[s] * beta * (lam - V_total)
        row = {'trial': t, 'lambda': lam}
        row.update({f'V_{s}': V[s] for s in stim})
        rows.append(row)
    return pd.DataFrame(rows)

# Simple acquisition: one CS (A), US present every trial
acq = rescorla_wagner([(['A'], 1.0)] * 30, {'A': 0.5})
print("Final V_A =", round(acq['V_A'].iloc[-1], 3))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(acq['trial'], acq['V_A'], marker='o', ms=3)
ax.axhline(1.0, color='red', ls='--', alpha=0.5, label='lambda (asymptote)')
ax.set_xlabel('Trial'); ax.set_ylabel('Associative strength V'); ax.set_title('Rescorla-Wagner: Acquisition')
ax.legend(); sns.despine(); plt.show()

### Task 2: Reproduce blocking, overshadowing, overexpectation, and conditioned inhibition

Each phenomenon falls out of the shared prediction-error term $(\lambda - V_{total})$. Build the trial schedules and confirm the signatures:

- **Blocking:** train A->US alone, then AB->US. B barely learns ($V_B \approx 0$).
- **Overshadowing:** train AB->US with $\alpha_A > \alpha_B$. A captures more of $\lambda$.
- **Overexpectation:** train A->US and B->US separately to asymptote, then AB->US together. Both $V$ values *decrease* because $V_{total} \approx 2\lambda$ overpredicts.
- **Conditioned inhibition:** interleave A->US with AB->no US. B acquires *negative* strength.

In [ ]:
# Blocking: A -> US (x20), then AB -> US (x20)
blocking = rescorla_wagner([(['A'], 1.0)] * 20 + [(['A', 'B'], 1.0)] * 20, {'A': 0.5, 'B': 0.5})

# Overshadowing: AB -> US with CS1 more salient than CS2
overshadow = rescorla_wagner([(['A', 'B'], 1.0)] * 30, {'A': 0.6, 'B': 0.2})

# Overexpectation: A->US and B->US separately (x20 each, interleaved), then AB->US together (x20)
overexp = rescorla_wagner([(['A'], 1.0), (['B'], 1.0)] * 20 + [(['A', 'B'], 1.0)] * 20,
                          {'A': 0.5, 'B': 0.5})

# Conditioned inhibition: A->US interleaved with AB->no US
condinhib = rescorla_wagner([(['A'], 1.0), (['A', 'B'], 0.0)] * 40, {'A': 0.5, 'B': 0.5})

print("Blocking      : V_B final  =", round(blocking['V_B'].iloc[-1], 3), "(blocked -> near 0)")
print("Overshadowing : V_A=", round(overshadow['V_A'].iloc[-1], 3),
      " V_B=", round(overshadow['V_B'].iloc[-1], 3))
print("Overexpect.   : V_A pre-compound =", round(overexp['V_A'].iloc[39], 3),
      " -> final =", round(overexp['V_A'].iloc[-1], 3))
print("Cond. inhib.  : V_B final  =", round(condinhib['V_B'].iloc[-1], 3), "(negative)")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes[0,0].plot(blocking['trial'], blocking['V_A'], label='V_A'); axes[0,0].plot(blocking['trial'], blocking['V_B'], label='V_B')
axes[0,0].axvline(20, color='gray', ls='--', alpha=0.5); axes[0,0].set_title('Blocking'); axes[0,0].legend()
axes[0,1].plot(overshadow['trial'], overshadow['V_A'], label='V_A (alpha=0.6)'); axes[0,1].plot(overshadow['trial'], overshadow['V_B'], label='V_B (alpha=0.2)')
axes[0,1].set_title('Overshadowing'); axes[0,1].legend()
axes[1,0].plot(overexp['trial'], overexp['V_A'], label='V_A'); axes[1,0].plot(overexp['trial'], overexp['V_B'], label='V_B')
axes[1,0].axvline(40, color='gray', ls='--', alpha=0.5); axes[1,0].set_title('Overexpectation'); axes[1,0].legend()
axes[1,1].plot(condinhib['trial'], condinhib['V_A'], label='V_A (excitor)'); axes[1,1].plot(condinhib['trial'], condinhib['V_B'], label='V_B (inhibitor)')
axes[1,1].axhline(0, color='black', lw=0.8); axes[1,1].set_title('Conditioned Inhibition'); axes[1,1].legend()
for ax in axes.flat: ax.set_xlabel('Trial'); ax.set_ylabel('V')
plt.tight_layout(); sns.despine(); plt.show()

## Part 2: The Mackintosh Model (Recursive Associability)

### Task 3: Implement the model

The Mackintosh (1975) model adds a dynamic *associability* term: a stimulus that predicts the US better than its competitors gains attention (its $\alpha$ rises), while a poorer predictor loses it. Implement the update rules for association strength $V$ and associability $\alpha$, then write simulation functions for (a) basic conditioning, (b) overshadowing (two CSs with different initial associability), and (c) blocking (CS1 trained alone, then CS1+CS2).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(73125)

class MackintoshModel:
    def __init__(self, theta=0.3):
        self.theta = theta  # Learning rate parameter

    def update_association(self, alpha, lambda_val, V_current):
        """Update association strength: dV = alpha * theta * (lambda - V)"""
        delta_V = alpha * self.theta * (lambda_val - V_current)
        return delta_V

    def update_associability(self, alpha, lambda_val, V_current, V_other=0):
        """
        Update associability based on prediction accuracy
        alpha(t+1) = alpha(t) + theta[|lambda - V_current| - |lambda - V_other|]
        """
        prediction_error_current = abs(lambda_val - V_current)
        prediction_error_other = abs(lambda_val - V_other)
        delta_alpha = self.theta * (prediction_error_current - prediction_error_other)
        return alpha + delta_alpha

def simulate_basic_conditioning(trials=25, alpha_init=0.5, theta=0.3, lambda_val=1.0):
    """Basic conditioning with one CS and one US"""
    model = MackintoshModel(theta)
    alpha = alpha_init
    V = 0.0
    data = []
    for trial in range(trials):
        delta_V = model.update_association(alpha, lambda_val, V)
        V += delta_V
        alpha = model.update_associability(alpha, lambda_val, V)
        alpha = np.clip(alpha, 0.01, 1.0)
        data.append({'trial': trial + 1, 'alpha': alpha, 'V': V,
                     'delta_V': delta_V, 'lambda': lambda_val, 'theta': theta})
    return pd.DataFrame(data)

def simulate_overshadowing(trials=30, alpha1_init=0.8, alpha2_init=0.3, theta=0.3, lambda_val=1.0):
    """Two CSs presented together; CS1 has higher initial associability and overshadows CS2."""
    model = MackintoshModel(theta)
    alpha1, alpha2 = alpha1_init, alpha2_init
    V1 = V2 = 0.0
    data = []
    for trial in range(trials):
        V_total = V1 + V2
        delta_V1 = model.update_association(alpha1, lambda_val, V_total)
        delta_V2 = model.update_association(alpha2, lambda_val, V_total)
        V1 += delta_V1
        V2 += delta_V2
        alpha1 = np.clip(model.update_associability(alpha1, lambda_val, V_total, V2), 0.01, 1.0)
        alpha2 = np.clip(model.update_associability(alpha2, lambda_val, V_total, V1), 0.01, 1.0)
        data.append({'trial': trial + 1, 'alpha1': alpha1, 'alpha2': alpha2,
                     'V1': V1, 'V2': V2, 'delta_V1': delta_V1, 'delta_V2': delta_V2,
                     'lambda': lambda_val, 'theta': theta})
    return pd.DataFrame(data)

def simulate_blocking(trials=40, alpha_init=0.5, theta=0.3, lambda_val=1.0):
    """CS1 trained alone first (Phase 1), then CS1+CS2 together (Phase 2). CS1 blocks CS2."""
    model = MackintoshModel(theta)
    alpha1 = alpha2 = alpha_init
    V1 = V2 = 0.0
    data = []
    for trial in range(trials):
        if trial < 20:
            delta_V1 = model.update_association(alpha1, lambda_val, V1)
            V1 += delta_V1
            alpha1 = np.clip(model.update_associability(alpha1, lambda_val, V1), 0.01, 1.0)
            data.append({'trial': trial + 1, 'phase': 'Phase1', 'alpha1': alpha1, 'alpha2': alpha2,
                         'V1': V1, 'V2': V2, 'delta_V1': delta_V1, 'delta_V2': 0,
                         'lambda': lambda_val, 'theta': theta})
        else:
            V_total = V1 + V2
            delta_V1 = model.update_association(alpha1, lambda_val, V_total)
            delta_V2 = model.update_association(alpha2, lambda_val, V_total)
            V1 += delta_V1
            V2 += delta_V2
            alpha1 = np.clip(model.update_associability(alpha1, lambda_val, V_total, V2), 0.01, 1.0)
            alpha2 = np.clip(model.update_associability(alpha2, lambda_val, V_total, V1), 0.01, 1.0)
            data.append({'trial': trial + 1, 'phase': 'Phase2', 'alpha1': alpha1, 'alpha2': alpha2,
                         'V1': V1, 'V2': V2, 'delta_V1': delta_V1, 'delta_V2': delta_V2,
                         'lambda': lambda_val, 'theta': theta})
    return pd.DataFrame(data)

### Task 4: Run the simulations and visualize

Run each simulation across a range of learning rates (theta), combine the results, and plot association strength and associability over trials. Confirm the expected signatures: overshadowing (CS1 gains more $V$ than CS2) and blocking (CS2 shows minimal learning in Phase 2).

In [ ]:
# Generate datasets across a range of learning rates (theta)
learning_rates = [0.1, 0.2, 0.3, 0.4, 0.5]

basic_datasets = []
for i, theta in enumerate(learning_rates):
    d = simulate_basic_conditioning(theta=theta); d['dataset_id'] = f'Basic_{i+1}'
    basic_datasets.append(d)

overshadowing_datasets = []
alpha1_values = [0.7, 0.8, 0.9, 0.85, 0.75]
alpha2_values = [0.2, 0.3, 0.1, 0.25, 0.15]
for i, theta in enumerate(learning_rates):
    d = simulate_overshadowing(theta=theta, alpha1_init=alpha1_values[i], alpha2_init=alpha2_values[i])
    d['dataset_id'] = f'Overshadowing_{i+1}'
    overshadowing_datasets.append(d)

blocking_datasets = []
for i, theta in enumerate(learning_rates):
    d = simulate_blocking(theta=theta); d['dataset_id'] = f'Blocking_{i+1}'
    blocking_datasets.append(d)

all_data = pd.concat(basic_datasets + overshadowing_datasets + blocking_datasets, ignore_index=True)
all_data.to_csv('mackintosh_model_data.csv', index=False)

fig, axes = plt.subplots(3, 2, figsize=(15, 12))
for d in basic_datasets:
    axes[0,0].plot(d['trial'], d['V'], label=f"theta={d['theta'].iloc[0]:.1f}", alpha=0.7)
    axes[0,1].plot(d['trial'], d['alpha'], label=f"theta={d['theta'].iloc[0]:.1f}", alpha=0.7)
axes[0,0].set_title('Basic Conditioning: Association Strength (V)'); axes[0,0].set_xlabel('Trial'); axes[0,0].set_ylabel('V'); axes[0,0].legend()
axes[0,1].set_title('Basic Conditioning: Associability (alpha)'); axes[0,1].set_xlabel('Trial'); axes[0,1].set_ylabel('alpha'); axes[0,1].legend()
for d in overshadowing_datasets:
    axes[1,0].plot(d['trial'], d['V1'], alpha=0.6, linestyle='-')
    axes[1,0].plot(d['trial'], d['V2'], alpha=0.6, linestyle='--')
axes[1,0].set_title('Overshadowing: V (solid=CS1, dashed=CS2)'); axes[1,0].set_xlabel('Trial'); axes[1,0].set_ylabel('V')
for d in overshadowing_datasets:
    axes[1,1].plot(d['trial'], d['alpha1'], alpha=0.6, linestyle='-')
    axes[1,1].plot(d['trial'], d['alpha2'], alpha=0.6, linestyle='--')
axes[1,1].set_title('Overshadowing: alpha (solid=CS1, dashed=CS2)'); axes[1,1].set_xlabel('Trial'); axes[1,1].set_ylabel('alpha')
for i, d in enumerate(blocking_datasets):
    p1, p2 = d[d['phase']=='Phase1'], d[d['phase']=='Phase2']
    axes[2,0].plot(p1['trial'], p1['V1'], color=f'C{i}', linestyle='-', alpha=0.7)
    axes[2,0].plot(p2['trial'], p2['V1'], color=f'C{i}', linestyle='--', alpha=0.7)
    axes[2,0].plot(p2['trial'], p2['V2'], color=f'C{i}', linestyle=':', alpha=0.7)
axes[2,0].axvline(x=20, color='gray', linestyle='-', alpha=0.4)
axes[2,0].set_title('Blocking: V (CS1 solid/dashed, CS2 dotted)'); axes[2,0].set_xlabel('Trial'); axes[2,0].set_ylabel('V')
for i, d in enumerate(blocking_datasets):
    p1, p2 = d[d['phase']=='Phase1'], d[d['phase']=='Phase2']
    axes[2,1].plot(p1['trial'], p1['alpha1'], color=f'C{i}', linestyle='-', alpha=0.7)
    axes[2,1].plot(p2['trial'], p2['alpha2'], color=f'C{i}', linestyle=':', alpha=0.7)
axes[2,1].axvline(x=20, color='gray', linestyle='-', alpha=0.4)
axes[2,1].set_title('Blocking: alpha'); axes[2,1].set_xlabel('Trial'); axes[2,1].set_ylabel('alpha')
plt.tight_layout(); plt.show()

print("Overshadowing final V1/V2 ratios:")
for i, d in enumerate(overshadowing_datasets):
    print(f"  theta={d['theta'].iloc[0]:.1f}: V1={d['V1'].iloc[-1]:.3f}, V2={d['V2'].iloc[-1]:.3f}")
print("\nBlocking final V2 (should stay low):")
for i, d in enumerate(blocking_datasets):
    print(f"  theta={d['theta'].iloc[0]:.1f}: V2={d['V2'].iloc[-1]:.3f}")

## Wrap-up

Compare the two models. The Rescorla-Wagner model explains blocking, overshadowing, overexpectation, and conditioned inhibition entirely through a shared prediction-error term with *fixed* saliences. What does the Mackintosh model add by letting associability change across trials, and which phenomena motivate that addition? Where in each update rule does the recursive structure $X(t+1) = f(X(t), I(t))$ appear?